<a href="https://colab.research.google.com/github/janithcyapa/DHCA-Framework/blob/main/Simulation/1.Thermal_Zone_Modeling_Verification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **RC Model Open-Loop Validation**


Before implementing active closed-loop control (such as Model Predictive Control), we must verify that our simplified physics-based Resistance-Capacitance (RC) model accurately mirrors the complex thermodynamic behavior of the EnergyPlus simulation environment.

This notebook cell executes an **open-loop validation test**. It works by running the full EnergyPlus simulation and simultaneously injecting its real-time boundary conditions—such as outdoor drybulb temperature ($T_{out}$), occupancy counts ($N_{occ}$), internal equipment loads ($Q_{equip}$), and HVAC supply air parameters—directly into our nonlinear RC system dynamics model at every timestep.

### 🎯 Key Objectives:

* 📉 **Quantify Model Discrepancy:** Compare the RC model's state predictions (Zone Air Temperature $T_{in}$, Mean Radiant Temperature $T_{m}$, Humidity Ratio $W_{in}$, and $CO_{2}$ concentration) against the ground-truth telemetry generated by EnergyPlus.
* ⚙️ **Verify System Coupling:** Ensure that the underlying differential equations accurately capture thermal inertia, zone mass flow exchange, and environmental heat transfers before deploying active controllers.

We can explore the data from this simulation run together to see how well they align.

### Installation And Setup Environment

In [22]:
# @title Setup Simulation Environment
# 1. Download the raw simulator.py file directly from your GitHub repository
!wget -q -O simulator.py https://raw.githubusercontent.com/janithcyapa/DHCA-Framework/refs/heads/main/Simulation/simulator.py

# 2. Import it exactly like a standard Python library
from simulator import *
print("✅ Simulator library successfully downloaded and loaded!")
SetupSimulationEnv()

✅ Simulator library successfully downloaded and loaded!
[SimEnv] : 🚀 Starting Environment Initialization...
[SimEnv] : 📦 Installing required pip packages...
[SimEnv] : ✅ Verified 'energy-plus-utility' version: 0.2.2+5
[SimEnv] : ⚙️ Configuring EnergyPlus backend binaries...
[SimEnv] : 🎉 EnergyPlus environment is ready.
[SimEnv] : 📚 Loading core modules into global namespace...
[SimEnv] : ✅ All dependencies successfully mapped and loaded.
[SimEnv] : 🌟 Environment Setup Complete! Ready for simulation.


### Zone Model Implementation

In [ ]:
# @title Setup Predictive RC Model (With Debugging)
def SetupPredictiveModel(sim, target_zone="SPACE1-1", debug_mode=True):
    import types
    import datetime
    import numpy as np
    import control as ct
    import traceback
    
    print(f"[SimEnv] : 🧠 Initializing Predictive RC Model Engine (DEBUG: {debug_mode})...")

    def zone_model(self, state):
        try:
            # Silently exit if E+ isn't ready, but log it if we are in extreme debug mode
            if not self.exchange.api_data_fully_ready(state):
                return
            if self.exchange.warmup_flag(state):
                return
                
            zone_id = target_zone

            # Time Stamping
            day = self.exchange.day_of_year(state)
            time = self.exchange.current_time(state)
            abs_time = (day * 24.0) + time

            if not hasattr(self, 'zones'):
                self.zones = {}
            if not hasattr(self, 'model_estimations'):
                self.model_estimations = {}

            if debug_mode:
                print(f"\n--- [DEBUG-RC] Timestep Triggered | Day: {day} | Hour: {time:.2f} ---")

            # --- Initialize Zone ---
            if zone_id not in self.zones:
                if debug_mode: print(f"[DEBUG-RC] First pass detected for {zone_id}. Generating handles...")
                raw_params = self.get_zone_thermal_parameters()[zone_id]
                
                handles = {
                    "T_in": self.exchange.get_variable_handle(state, "Zone Mean Air Temperature", zone_id),
                    "T_m": self.exchange.get_variable_handle(state, "Zone Mean Radiant Temperature", zone_id),
                    "W_in": self.exchange.get_variable_handle(state, "Zone Mean Air Humidity Ratio", zone_id),
                    "CO2_in": self.exchange.get_variable_handle(state, "Zone Air CO2 Concentration", zone_id),
                    "N_occ": self.exchange.get_variable_handle(state, "Zone People Occupant Count", zone_id),
                    "T_out": self.exchange.get_variable_handle(state, "Zone Outdoor Air Drybulb Temperature", zone_id),
                    "Q_equip": self.exchange.get_variable_handle(state, "Zone Electric Equipment Total Heating Rate", zone_id),
                    "m_dot": self.exchange.get_variable_handle(state, "System Node Mass Flow Rate", f"{zone_id} PTAC SUPPLY INLET"),
                    "T_s": self.exchange.get_variable_handle(state, "System Node Temperature", f"{zone_id} PTAC SUPPLY INLET"),
                    "W_s": self.exchange.get_variable_handle(state, "System Node Humidity Ratio", f"{zone_id} PTAC SUPPLY INLET"),
                    "C_s": self.exchange.get_variable_handle(state, "System Node CO2 Concentration", f"{zone_id} PTAC SUPPLY INLET"),
                }

                # --- VALIDATE HANDLES ---
                if debug_mode:
                    missing_handles = [k for k, v in handles.items() if v <= 0] # 0 or -1 usually indicates missing
                    if missing_handles:
                        print(f"⚠️ [DEBUG-WARN] Missing IDF Outputs for {zone_id}: {missing_handles}")
                        print("    Ensure these are defined in your Output:Variable list in the IDF!")

                inv_R_env_ext = 0.0
                R_env_gnd = None
                adj_zones = []

                for b in raw_params["boundaries"]:
                    target = b["target"]
                    r_abs = float(b["R_absolute_K_W"])
                    if target == "Ground": R_env_gnd = r_abs
                    elif target == "Environment" or b["boundary_condition"] == "outdoors": inv_R_env_ext += (1.0 / r_abs)
                    else: adj_zones.append({ "zone": target, "R_env": r_abs, "handle_T_in": self.exchange.get_variable_handle( state, "Zone Mean Air Temperature", target )})
                R_env_ext = 1.0 / inv_R_env_ext if inv_R_env_ext > 0 else float('inf')

                # --- Core Dynamics ---
                def _dynamics(t, x, u, params):
                    T_in, T_m, W_in, C_in = x 
                    V_dot_s = float(u[0]) 

                    rho_air, cp_air = 1.204, 1006.0
                    q_person, g_w_person, g_co2_person = 100.0, 5e-5, 1e-5
                    R_env_ext = float(params.get('R_env_ext', float('inf')))
                    R_env_gnd = float(params.get('R_env_gnd', float('inf')))
                    R_int = float(params['R_int'])
                    C_air = float(params['C_air'])
                    C_mass = float(params['C_mass'])
                    M_air = float(params['M_air'])
                    V_room = float(params['V_room'])

                    T_s, W_s, C_s = params['T_s'], params['W_s'], params['C_s']
                    N_occ, Q_equip, T_out = params['N_occ'], params['Q_equip'], params['T_out']
                    d_T, d_W, d_C = params['d_T'], params['d_W'], params['d_C'] 
                    
                    q_env = (T_out - T_in) / R_env_ext if R_env_ext < float('inf') else 0.0
                    q_gnd = (22.0 - T_in) / R_env_gnd if R_env_gnd < float('inf') else 0.0
                    q_adj = sum([(float(adj['T_in']) - float(T_in)) / float(adj['R_env']) for adj in params['adj_zones']])
                        
                    q_mass = (T_m - T_in) / R_int
                    q_int = (N_occ * q_person) + Q_equip
                    q_s = rho_air * V_dot_s * cp_air * (T_s - T_in)
                    
                    dT_in_dt = (q_env + q_gnd + q_adj + q_mass + q_int + q_s + d_T) / C_air
                    dT_m_dt = (T_in - T_m) / ( C_mass * R_int)
                    
                    dot_m_s = rho_air * V_dot_s
                    dW_in_dt = (N_occ * g_w_person + dot_m_s * (W_s - W_in) + d_W) / M_air
                    dC_in_dt = (N_occ * g_co2_person + V_dot_s * (C_s - C_in) + d_C) / V_room

                    return np.array([dT_in_dt, dT_m_dt, dW_in_dt, dC_in_dt], dtype=float).flatten()

                def _outputs(t, x, u, params):
                    return [x[0], x[1], x[2], x[3]]

                sys_ode = ct.NonlinearIOSystem(
                    _dynamics, _outputs,
                    inputs=['V_dot_s'],
                    outputs=['T_in_obs','T_m_obs', 'W_in_obs', 'C_in_obs'],
                    states=['T_in', 'T_m', 'W_in', 'C_in'],
                    name=f'sys_{zone_id}'
                )

                self.zones[zone_id] = types.SimpleNamespace(
                    last_time=abs_time, V_room=float(raw_params['V_room']),
                    M_air=float(raw_params['M_air']), C_air=float(raw_params['C_air']),
                    C_mass=float(raw_params['C_mass']), R_int=float(raw_params['R_int']),
                    R_env_gnd=R_env_gnd, R_env_ext=R_env_ext, adj_zones=adj_zones,
                    handles=handles, sys_ode=sys_ode
                )
                print(f"[Model]  : ✅ System Dynamics initialized for {zone_id}")

            else:
                self.zones[zone_id].last_time = abs_time

            z = self.zones[zone_id]
            
            # Runtime Variable Fetching
            m_dot_current = self.exchange.get_variable_value(state, z.handles["m_dot"])
            v_dot_current = m_dot_current / 1.204 
            u_current = [v_dot_current]
            
            current_adj_zones = [{
                'T_in': self.exchange.get_variable_value(state, adj["handle_T_in"]),
                'R_env': adj["R_env"]
            } for adj in z.adj_zones]

            current_params = {
                'C_air': z.C_air, 'C_mass': z.C_mass,
                'R_env_ext': z.R_env_ext, 'R_env_gnd':z.R_env_gnd,
                'R_int': z.R_int, 'M_air': z.M_air, 'V_room': z.V_room,
                'T_out': self.exchange.get_variable_value(state, z.handles["T_out"]), 
                'N_occ': self.exchange.get_variable_value(state, z.handles["N_occ"]),
                'Q_equip': self.exchange.get_variable_value(state, z.handles["Q_equip"]),
                'T_s': self.exchange.get_variable_value(state, z.handles["T_s"]),
                'W_s': self.exchange.get_variable_value(state, z.handles["W_s"]),
                'C_s': self.exchange.get_variable_value(state, z.handles["C_s"]),
                'd_T': 0.0, 'd_W': 0.0, 'd_C': 0.0,
                'adj_zones': current_adj_zones 
            }

            x_solver = [
                self.exchange.get_variable_value(state, z.handles["T_in"]),
                self.exchange.get_variable_value(state, z.handles["T_m"]),
                self.exchange.get_variable_value(state, z.handles["W_in"]),
                self.exchange.get_variable_value(state, z.handles["CO2_in"])
            ]

            dt_hours = self.exchange.system_time_step(state)
            if dt_hours == 0: 
                dt_hours = self.exchange.zone_time_step(state)
            time_vector = [0, dt_hours * 3600.0]

            if debug_mode:
                print(f"[DEBUG-RC] dt_hours = {dt_hours:.4f} | time_vector = {time_vector}")
                print(f"[DEBUG-RC] Initial States (X0): T_in={x_solver[0]:.2f}, T_m={x_solver[1]:.2f}, W_in={x_solver[2]:.5f}, CO2={x_solver[3]:.1f}")
                print(f"[DEBUG-RC] Control Inputs (U): m_dot={m_dot_current:.4f}, V_dot={v_dot_current:.4f}")
                print(f"[DEBUG-RC] Dynamic Params: T_out={current_params['T_out']:.2f}, N_occ={current_params['N_occ']}, Q_eq={current_params['Q_equip']:.1f}")

            # Solve the ODE
            response = ct.input_output_response(z.sys_ode, time_vector, U=u_current, X0=x_solver, params=current_params)
            x_predicted_next = response.states[:, -1]

            if debug_mode:
                print(f"[DEBUG-RC] Pred Next States: T_in_pred={x_predicted_next[0]:.2f}, T_m_pred={x_predicted_next[1]:.2f}")

            # Save the estimation for the State Logger
            self.model_estimations[zone_id] = {
                "T_in_pred": x_predicted_next[0],
                "T_m_pred": x_predicted_next[1],
                "W_in_pred": x_predicted_next[2],
                "C_in_pred": x_predicted_next[3]
            }

        except Exception as e:
            print(f"\n[Model]   : ❌ [FATAL PREDICTION CRASH] {e}")
            if debug_mode:
                traceback.print_exc()

    sim.zone_model = types.MethodType(zone_model, sim)
    sim.register_handlers("begin", [{"method_name": "zone_model"}])
    print(f"[SimEnv] : ✅ Predictive RC Model registered on 'begin' hook.")

### Simualtion

In [23]:
# @title Run Complete Simualtion
sim = SetSimulationModel(verbose=0)
ModifySimulationModel(sim)
SetupStateLogger(sim)
SetupOccupancyInjector(sim)
SetupActuatorController(sim)
SetupPredictiveModel(sim,'SPACE1-1', True )
RunSimulation(sim, num_days=3, rate=12)
df_log,csv  = ExtractSimulationData(sim, "model_validation_sim.csv")

[SimEnv] : ⚙️ Setting up Simulation Model...
[SimEnv] : 📂 Using EnergyPlus structural binary path: /root/EnergyPlus-25-1-0
[SimEnv] : 📋 Copying baseline IDF template to workspace...
[SimEnv] : 🌐 Fetching remote climatological data (EPW)...
[SimEnv] : ⚡ Launching ExpandObjects to compile macro templates...
[SimEnv] : ✅ Successfully generated expanded.idf component definitions.
[SimEnv] : 🔗 Injecting compiled model and weather runtime tracks into simulation engine...
[SimEnv] : 🎉 Simulation Model Environment completely configured and linked!
[SimEnv] : 🔧 Beginning Simulation Model Custom Modifications...
[SimEnv] : 💉 Injecting standalone humidification macro-blocks into all zones...
[SimRun] : 🛑 Executing environment Dry Run...
[SimEnv] : ✅ Standalone Humidifiers successfully injected globally.
[SimEnv] : 🌟 Model Modification Stage Complete!
[SimEnv] : 📡 Initializing and binding Diagnostic State Logger...
[SimEnv] : ✅ Diagnostic Logger globally registered and armed.
[SimEnv] : 👥 Initiali

,timestamp,day,hour,minute,time_decimal,T_out,W_out,RH_out_%,CO2_out,SPACE1-1_T_in,...,SPACE3-1_W_in_pred,SPACE3-1_C_in_pred,SPACE4-1_T_in_pred,SPACE4-1_T_m_pred,SPACE4-1_W_in_pred,SPACE4-1_C_in_pred,SPACE5-1_T_in_pred,SPACE5-1_T_m_pred,SPACE5-1_W_in_pred,SPACE5-1_C_in_pred
Time_Hours,,,,,,,,,,,,,,,,,,,,,
0.083333,1767225900,1,0,5,0.083333,25.293617,NaN,83.916667,420.0,23.095285,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
0.166667,1767226200,1,0,10,0.166667,25.185283,NaN,84.833333,420.0,23.100706,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
0.250000,1767226500,1,0,15,0.250000,25.076950,NaN,85.750000,420.0,23.116144,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
0.333333,1767226800,1,0,20,0.333333,24.968617,NaN,86.666667,420.0,23.127456,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
0.416667,1767227040,1,0,24,0.416667,24.860283,NaN,87.583333,420.0,23.129871,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
71.666667,1767483600,3,23,40,23.666667,25.001950,NaN,89.666667,420.0,23.597609,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
71.750000,1767483900,3,23,45,23.750000,25.001950,NaN,89.750000,420.0,23.597303,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
71.833333,1767484200,3,23,50,23.833333,25.001950,NaN,89.833333,420.0,23.591923,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [24]:
# @title Plot Data
import plotly.io as pio
pio.renderers.default = "vscode"
layout_config = {
        "global_title": "RC Model Estimation vs Actual Reality: {target_zone}",
        "height": 1200,
        "template": "plotly_dark",
        "vertical_spacing": 0.05,
        "subplots": [
            {
                "title": "1. Zone Air Temperature (T_in)",
                "y_title": "Temp (°C)",
                "group_id": "temp_air",
                "traces": [
                    {"col": "T_out", "name": "Ambient (T_out)", "color": "white", "dash": "dash"},
                    {"col": "{target_zone}_T_in", "name": "Actual (E+)", "color": "#00e5ff", "width": 2},
                    {"col": "{target_zone}_T_in_pred", "name": "Predicted (RC Model)", "color": "#ff00ff", "dash": "dot", "width": 3}
                ]
            },
            {
                "title": "2. Radiant Thermal Mass Temperature (T_m)",
                "y_title": "Temp (°C)",
                "group_id": "temp_mass",
                "traces": [
                    {"col": "{target_zone}_T_m", "name": "Actual (E+)", "color": "#00e5ff", "width": 2},
                    {"col": "{target_zone}_T_m_pred", "name": "Predicted (RC Model)", "color": "#ff00ff", "dash": "dot", "width": 3}
                ]
            },
            {
                "title": "3. Humidity Ratio (W_in)",
                "y_title": "W (kg/kg)",
                "group_id": "humidity",
                "traces": [
                    {"col": "W_out", "name": "Ambient (W_out)", "color": "white", "dash": "dash"},
                    {"col": "{target_zone}_W_in", "name": "Actual (E+)", "color": "#00e5ff", "width": 2},
                    {"col": "{target_zone}_W_in_pred", "name": "Predicted (RC Model)", "color": "#ff00ff", "dash": "dot", "width": 3}
                ]
            },
            {
                "title": "4. CO2 Concentration (C_in)",
                "y_title": "CO2 (ppm)",
                "group_id": "co2",
                "traces": [
                    {"col": "CO2_out", "name": "Ambient (C_out)", "color": "white", "dash": "dash"},
                    {"col": "{target_zone}_CO2_in", "name": "Actual (E+)", "color": "#00e5ff", "width": 2},
                    {"col": "{target_zone}_C_in_pred", "name": "Predicted (RC Model)", "color": "#ff00ff", "dash": "dot", "width": 3}
                ]
            }
        ]
    }
fig = GenerateSimulationPlots(df_log, target_zone="SPACE1-1",layout_config=layout_config)

[Plot] : 📊 Initializing modular telemetry plotting engine for: SPACE1-1
[Plot] : ✅ Canvas rendering complete. Dispatched dashboard view.
